In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "cells": [
  {
   "cell_type": "markdown",
   "id": "cell-01",
   "metadata": {},
   "source": [
    "# baza500 — FAQ Scraper: 5 polskich sklepów meblowych\n",
    "\n",
    "**Cel:** Zebranie 500 unikalnych par pytanie/odpowiedź z kart FAQ 5 (docelowo 20) sklepów meblowych w Polsce.\n",
    "\n",
    "## Zbadane sklepy i struktury HTML\n",
    "\n",
    "| Sklep | URL FAQ | Pytanie (selektor) | Odpowiedź (selektor) |\n",
    "|---|---|---|---|\n",
    "| **Mebligo** | `/content/9-najczesciej-zadawane-pytania-faq` | `div[id^=\"ac-\"] button[id^=\"ac-trigger-\"]` | następne rodzeństwo `div` w bloku akordeonu |\n",
    "| **Stolar Meble** | `/faq/` | `button` w `div#content` | następny element po `button` |\n",
    "| **MeblujemyDOM** | `/pl/help/faq-najczesciej-zadawane-pytania-2` | elementy z `?` w `div#content` (nie-linki) | kolejne bloki tekstowe do następnego `?` |\n",
    "| **SalonMeblowy.net** | `/faq.ehtml` | `h2`, `h3` w `div#afaq` | kolejne `p`/`div` do następnego nagłówka |\n",
    "| **MebleM4** | `/faq-najczesciej-zadawane-pytania-w-meblem4-pl,p39.html` | elementy pasujące do `^\\d+\\.\\s+` | kolejne bloki tekstowe do następnego numeru |\n"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-02",
   "metadata": {},
   "source": [
    "## Instalacja zależności"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-03",
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip install requests beautifulsoup4 pandas rapidfuzz"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-04",
   "metadata": {},
   "source": [
    "## Importy i konfiguracja"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-05",
   "metadata": {},
   "outputs": [],
   "source": [
    "import re\n",
    "import requests\n",
    "from bs4 import BeautifulSoup\n",
    "import pandas as pd\n",
    "\n",
    "HEADERS = {\n",
    "    \"User-Agent\": (\n",
    "        \"Mozilla/5.0 (Windows NT 10.0; Win64; x64) \"\n",
    "        \"AppleWebKit/537.36 (KHTML, like Gecko) \"\n",
    "        \"Chrome/124.0.0.0 Safari/537.36\"\n",
    "    )\n",
    "}\n",
    "\n",
    "def get_soup(url: str) -> BeautifulSoup:\n",
    "    r = requests.get(url, headers=HEADERS, timeout=20)\n",
    "    r.raise_for_status()\n",
    "    return BeautifulSoup(r.text, \"html.parser\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-06",
   "metadata": {},
   "source": [
    "## 1. Mebligo.pl\n",
    "\n",
    "**Struktura:** akordeon `div[id^=\"ac-N\"]` → `button[id^=\"ac-trigger-N\"]` (pytanie) + ukryty `div` z odpowiedzią (obecny w HTML mimo CSS display:none)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-07",
   "metadata": {},
   "outputs": [],
   "source": [
    "def parse_mebligo() -> list:\n",
    "    url = \"https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq\"\n",
    "    soup = get_soup(url)\n",
    "    qas = []\n",
    "\n",
    "    for block in soup.find_all(\"div\", id=re.compile(r\"^ac-\\d+$\")):\n",
    "        btn = block.find(\"button\", id=re.compile(r\"^ac-trigger-\"))\n",
    "        if not btn:\n",
    "            continue\n",
    "        question = btn.get_text(\" \", strip=True).rstrip(\" +\").strip()\n",
    "\n",
    "        answer_parts = []\n",
    "        for child in block.children:\n",
    "            if hasattr(child, \"name\") and child.name and child != btn.parent:\n",
    "                text = child.get_text(\" \", strip=True)\n",
    "                if text:\n",
    "                    answer_parts.append(text)\n",
    "        answer = \" \".join(answer_parts).strip()\n",
    "\n",
    "        if question and answer:\n",
    "            qas.append({\"shop\": \"Mebligo\", \"source_url\": url,\n",
    "                        \"question\": question, \"answer\": answer})\n",
    "    return qas\n",
    "\n",
    "r = parse_mebligo()\n",
    "print(f\"Mebligo: {len(r)} Q&A\")\n",
    "r[:2]"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-08",
   "metadata": {},
   "source": [
    "## 2. StolarMeble.pl\n",
    "\n",
    "**Struktura:** `div#content` → `button` (pytanie) + następny element rodzeństwo (odpowiedź). Wszystkie elementy widoczne bezpośrednio w DOM."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-09",
   "metadata": {},
   "outputs": [],
   "source": [
    "def parse_stolar() -> list:\n",
    "    url = \"https://stolarmeble.pl/faq/\"\n",
    "    soup = get_soup(url)\n",
    "    qas = []\n",
    "\n",
    "    content = soup.find(\"div\", id=\"content\")\n",
    "    if not content:\n",
    "        return qas\n",
    "\n",
    "    for btn in content.find_all(\"button\"):\n",
    "        question = btn.get_text(\" \", strip=True)\n",
    "        if not question:\n",
    "            continue\n",
    "        answer_node = btn.find_next_sibling()\n",
    "        if not answer_node:\n",
    "            answer_node = btn.parent.find_next_sibling()\n",
    "        answer = answer_node.get_text(\" \", strip=True) if answer_node else \"\"\n",
    "\n",
    "        if question and answer:\n",
    "            qas.append({\"shop\": \"Stolar Meble\", \"source_url\": url,\n",
    "                        \"question\": question, \"answer\": answer})\n",
    "    return qas\n",
    "\n",
    "r = parse_stolar()\n",
    "print(f\"Stolar: {len(r)} Q&A\")\n",
    "r[:2]"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-10",
   "metadata": {},
   "source": [
    "## 3. MeblujemyDOM.pl\n",
    "\n",
    "**Struktura:** `div#content` → naprzemienne bloki tekstowe: element z `?` = pytanie, kolejne bloki do następnego `?` = odpowiedź."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-11",
   "metadata": {},
   "outputs": [],
   "source": [
    "def parse_meblujemydom() -> list:\n",
    "    url = \"https://meblujemydom.pl/pl/help/faq-najczesciej-zadawane-pytania-2\"\n",
    "    soup = get_soup(url)\n",
    "    qas = []\n",
    "\n",
    "    content = soup.find(\"div\", id=\"content\")\n",
    "    if not content:\n",
    "        return qas\n",
    "\n",
    "    children = [c for c in content.children\n",
    "                if hasattr(c, \"name\") and c.name is not None]\n",
    "\n",
    "    i = 0\n",
    "    while i < len(children):\n",
    "        node = children[i]\n",
    "        text = node.get_text(\" \", strip=True)\n",
    "        is_toc_link = bool(node.find(\"a\")) and re.match(r\"^\\d+\\.\", text)\n",
    "        is_question = (\n",
    "            \"?\" in text\n",
    "            and not is_toc_link\n",
    "            and 10 < len(text) < 300\n",
    "            and not node.find(\"a\", href=True)\n",
    "        )\n",
    "        if is_question:\n",
    "            answer_parts = []\n",
    "            j = i + 1\n",
    "            while j < len(children):\n",
    "                next_text = children[j].get_text(\" \", strip=True)\n",
    "                if \"?\" in next_text and len(next_text) < 300:\n",
    "                    break\n",
    "                if next_text:\n",
    "                    answer_parts.append(next_text)\n",
    "                j += 1\n",
    "            answer = \" \".join(answer_parts).strip()\n",
    "            if text and answer:\n",
    "                qas.append({\"shop\": \"MeblujemyDOM\", \"source_url\": url,\n",
    "                            \"question\": text, \"answer\": answer})\n",
    "            i = j\n",
    "        else:\n",
    "            i += 1\n",
    "    return qas\n",
    "\n",
    "r = parse_meblujemydom()\n",
    "print(f\"MeblujemyDOM: {len(r)} Q&A\")\n",
    "r[:2]"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "cell-12",
   "metadata": {},
   "source": [
    "## 4. SalonMeblowy.net.pl\n",
    "\n",
    "**Struktura:** `div#afaq` → `h2`/`h3` = pytanie, kolejne `p`/`div`/`generic` do następnego nagłówka = odpowiedź."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "cell-13",
   "metadata": {},
   "outputs": [],
   "source": [
    "def parse_salonmeblowy() -> list:\n",
    "    url = \"https://www.salonmeblowy.net.pl/faq.ehtml\"\n",
    "    soup = get_soup(url)\n",
    "    qas = []\n",
    "\n",
    "    afaq = soup.find(\"div\", id=\"afaq\")\n",
    "    if not afaq:\n",
    "        return qas\n",
    "\n",
    "    children = [c for c in afaq.children\n",
    "                if hasattr(c, \"name\") and c.name is not None]\n",
    "\n",
    "    i = 0\n",
    "    while i < len(children):\n",
    "        node = children[i]\n",
    "        if node.name in (\"h2\", \"h3\"):\n",
    "            question = node.get_text(\" \", strip=True)\n",
    "            answer_parts = []\n",
    "            j = i + 1\n",
    "            while j < len(children):\n",
    "                sibling = children[j]\n",
    "                if sibling.name in (\"h2\", \"h3\"):\n",
    "                    break\n",
    "                text = sibling.get_text(\" \", strip=True)\n",